# D2 Aaya — Online Retrieval Adaptation Layer

**Boundary statement:** This notebook uses **Salma’s static hybrid retriever** as the baseline/dependency. **Aaya’s contribution is the online/adaptive routing layer**, not the implementation of BM25, dense, or hybrid retrieval.

**Required sentence for D2 boundary clarity:** This notebook does not re-implement Salma’s retrieval evaluation. It uses the hybrid retriever as a fixed baseline and evaluates whether an online River/FeedbackAdapter layer changes retrieval behavior over time.

This notebook evaluates three routing conditions on top of the same hybrid retrieval dependency:

1. **Static hybrid baseline** — the chosen hybrid retriever is used with fixed BM25/dense weights.
2. **Topic-gated hybrid routing** — River predicts the query topic, then the router applies a topic-specific BM25/dense profile.
3. **Adaptive hybrid routing** — River predicts the topic, then `FeedbackAdapter` updates topic-specific BM25/dense weights over the stream.

The notebook’s claim is therefore **not** “Aaya built the retriever.” The claim is:

> Aaya connected the D1 River online learner and the FeedbackAdapter to the retrieval path and measured whether this online layer changes retrieval behavior compared with the fixed static hybrid baseline.


## Boundary with Salma’s work

- **Salma:** builds and evaluates the retrieval methods, including BM25-only, dense-only, hybrid retrieval, retrieval metrics, and `d2_search_metrics.csv`.
- **Aaya:** adds online topic prediction and feedback-adaptive weighting **on top of the chosen hybrid retriever**.
- This notebook imports retrieval classes only to instantiate Salma’s hybrid retriever as a fixed dependency for the online-routing experiment.
- Aaya’s required output remains `reports/tables/d2_online_vs_static.csv`.


## 1. Why D1 feedback matters

The D1 issue was that the online learner proved it could classify query topics and detect drift, but it did **not** influence retrieval. For D2, the online component must sit **inside the retrieval route**:

```text
User query
   ↓
River topic classifier predicts topic
   ↓
Routing decision chooses retrieval weights / index / filters
   ↓
Hybrid retrieval returns evidence
   ↓
Answer generation
   ↓
Feedback updates adapter and/or online classifier
```

So the D2 claim becomes: **online learning improves or adapts retrieval behavior**, not only topic classification accuracy.

## 2. Two integration designs

### Design A — Topic-gated hybrid retrieval

**Idea:** use the River topic classifier as a router. For each query, predict the climate topic, then use a topic-specific retrieval setting.

Example:

| Predicted topic | Routing behavior |
|---|---|
| `policy_governance` | Higher BM25 weight because exact policy names, NDCs, and regulations matter |
| `climate_science` | Higher dense weight because semantic scientific explanations matter |
| `uae_cop28` | Balanced or BM25-heavy because exact COP28 phrases and source names matter |

**Strength:** easy to implement for D2 because it only changes retrieval parameters before calling the existing retriever.

**Weakness:** it adapts only through topic prediction, not through user feedback unless combined with Design B.

---

### Design B — Feedback-adaptive hybrid retrieval

**Idea:** after retrieval/answering, explicit or simulated feedback updates topic-specific BM25/dense weights. If the answer missed exact policy names or citations, BM25 increases. If lexical matching was too narrow, dense retrieval increases.

```text
query → topic prediction → current topic weights → retrieval → feedback → update topic weights
```

**Strength:** directly satisfies online adaptation because retrieval weights change over time.

**Weakness:** needs feedback labels. For D2, these can be simulated from a gold set or collected using a simple thumbs-up/down UI.

---

### Feasible D2 choice

The strongest feasible D2 design is a **combined router**:

> **Topic-gated adaptive hybrid retrieval** = River predicts the topic, then `FeedbackAdapter` provides/upgrades the BM25 and dense weights for that topic.

This is feasible because it reuses the existing D1 code: the River classifier already predicts climate topics, and the feedback adapter already stores per-topic hybrid weights. The only new D2 work is adding a wrapper around retrieval.

## 3. Aaya metrics table to produce

This notebook reports **online-routing/adaptation metrics**, not Salma’s BM25-vs-dense-vs-hybrid retrieval comparison.

| Method | Dependency | Online layer | BM25 weight source | Recall@5 | MRR@5 | nDCG@5 | Answer citation hit rate | Prequential topic accuracy | Mean latency ms | Notes |
|---|---|---|---:|---:|---:|---:|---:|---:|---:|---|
| Static hybrid baseline | Salma hybrid retriever | none | fixed 0.50 / 0.50 |  |  |  |  | N/A |  | baseline |
| Topic-gated hybrid routing | Salma hybrid retriever | River topic prediction | fixed topic profile |  |  |  |  |  |  | routing only |
| Adaptive hybrid routing | Salma hybrid retriever | River + FeedbackAdapter | adaptive per-topic weights |  |  |  |  |  |  | routing + adaptation |

Primary metrics:

- **Recall@5:** whether at least one relevant-topic chunk appears in top 5.
- **MRR@5:** how early the first relevant-topic result appears.
- **nDCG@5:** ranking quality across the top 5.
- **Answer citation hit rate proxy:** whether the top result matches the expected topic.
- **Prequential topic accuracy:** confirms the online learner is evaluated predict-then-learn.
- **Mean latency:** checks whether the online routing layer adds runtime cost.

The required Aaya comparison file is:

```text
reports/tables/d2_online_vs_static.csv
```


In [1]:
# 4. Setup
from __future__ import annotations

import math
import random
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Optional
import json
import sys

import pandas as pd

# Resolve the project root safely whether this notebook runs from notebooks/ or project root.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() in {"notebooks", "notebook"}:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# D1 online-learning components used by Aaya.
RIVER_FILE = PROJECT_ROOT / "src" / "learning" / "river_topic_classifier.py"
FEEDBACK_FILE = PROJECT_ROOT / "src" / "learning" / "feedback_adapter.py"

print("Project root:", PROJECT_ROOT)
print("River file exists:", RIVER_FILE.exists())
print("Feedback file exists:", FEEDBACK_FILE.exists())


Project root: D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent
River file exists: True
Feedback file exists: True


In [2]:
%pip install river

[local-executor] skipped notebook magic: %pip install river


In [3]:
# 5. Load the uploaded D1 components dynamically
# This avoids needing the exact original package structure.
# The uploaded River file imports drift_detector.py. If that file is not present
# in this standalone notebook environment, we provide a tiny compatible fallback.

import importlib.util
import sys
import types

if "drift_detector" not in sys.modules:
    drift_detector_stub = types.ModuleType("drift_detector")

    class ADWINDriftDetector:
        def __init__(self, delta=0.01, min_num_instances=250):
            self.delta = delta
            self.min_num_instances = min_num_instances
            self.alerts = []
            self.values = []

        def update(self, correct: bool, index: int) -> bool:
            # Standalone fallback only. The real project should use the original ADWIN helper.
            self.values.append(1 if correct else 0)
            return False

        def get_alert_indices(self):
            return self.alerts

    drift_detector_stub.ADWINDriftDetector = ADWINDriftDetector
    sys.modules["drift_detector"] = drift_detector_stub


def load_module_from_path(module_name: str, file_path: Path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module

river_mod = load_module_from_path("river_topic_classifier_d2", RIVER_FILE)
feedback_mod = load_module_from_path("feedback_adapter_d2", FEEDBACK_FILE)

CLIMATE_TOPICS = river_mod.CLIMATE_TOPICS
FeedbackAdapter = feedback_mod.FeedbackAdapter

print("Topics:", CLIMATE_TOPICS)

Topics: ['mitigation', 'adaptation', 'climate_science', 'policy_governance', 'technology_innovation', 'uae_cop28']


In [4]:
# 6. Topic profiles for topic-gated retrieval
# These are simple D2 routing rules. Tune them after running the real retrieval evaluation.

TOPIC_ROUTING_PROFILES = {
    "mitigation": {"bm25_weight": 0.55, "dense_weight": 0.45},
    "adaptation": {"bm25_weight": 0.45, "dense_weight": 0.55},
    "climate_science": {"bm25_weight": 0.40, "dense_weight": 0.60},
    "policy_governance": {"bm25_weight": 0.65, "dense_weight": 0.35},
    "technology_innovation": {"bm25_weight": 0.45, "dense_weight": 0.55},
    "uae_cop28": {"bm25_weight": 0.60, "dense_weight": 0.40},
}

STATIC_WEIGHTS = {"bm25_weight": 0.50, "dense_weight": 0.50}

pd.DataFrame([
    {"topic": topic, **weights}
    for topic, weights in TOPIC_ROUTING_PROFILES.items()
])

In [5]:
# 7. Project path confirmation

print("Project root:", PROJECT_ROOT)
print("Project root in sys.path:", str(PROJECT_ROOT) in sys.path)


Project root: D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent
Project root in sys.path: True


In [6]:
chunks_path = PROJECT_ROOT / "data" / "sample" / "sample_chunks.json"

print("Chunks path:", chunks_path)
print("Exists:", chunks_path.exists())

with open(chunks_path, "r", encoding="utf-8") as f:
    chunks = json.load(f)

for i, chunk in enumerate(chunks):
    chunk.setdefault("chunk_id", f"chunk_{i}")

Chunks path: D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent\data\sample\sample_chunks.json
Exists: True


In [7]:
# 8. Instantiate Salma's hybrid retriever as Aaya's fixed baseline dependency
#
# Boundary note:
# - These imports are NOT Aaya's retrieval contribution.
# - BM25Retriever, NumpyDenseRetriever, and HybridRetriever belong to the static retrieval stack.
# - Aaya uses the completed hybrid retriever only as the baseline/dependency.
# - Aaya's contribution begins when the online router changes bm25_weight over time.

from src.retrieval.bm25_retriever import BM25Retriever
from src.retrieval.dense_retriever import NumpyDenseRetriever
from src.retrieval.hybrid_retriever import HybridRetriever

print("Loaded Salma's retrieval classes as fixed baseline dependencies.")

bm25_retriever = BM25Retriever(chunks)
dense_retriever = NumpyDenseRetriever(chunks)

hybrid = HybridRetriever(
    bm25_retriever=bm25_retriever,
    dense_retriever=dense_retriever,
    bm25_weight=0.5,
    normalization="minmax",   # important because adaptive weights require weighted fusion
)

print("Instantiated Salma's hybrid retriever baseline for Aaya's online-routing experiment.")


Loaded Salma's retrieval classes as fixed baseline dependencies.
Instantiated Salma's hybrid retriever baseline for Aaya's online-routing experiment.


In [8]:
import inspect

print("Loaded HybridRetriever:", type(hybrid))
print("hybrid.search signature:", inspect.signature(hybrid.search))
print("Has bm25_weight attribute:", hasattr(hybrid, "bm25_weight"))
print("Has normalization attribute:", hasattr(hybrid, "normalization"))

if hasattr(hybrid, "normalization"):
    print("Current normalization:", hybrid.normalization)

Loaded HybridRetriever: <class 'src.retrieval.hybrid_retriever.HybridRetriever'>
hybrid.search signature: (query: str, k: int = 5, filters: dict | None = None)
Has bm25_weight attribute: True
Has normalization attribute: True
Current normalization: minmax


In [9]:
# 9. Thin wrapper around Salma's hybrid retriever

import inspect

def retrieve_hybrid(
    query: str,
    bm25_weight: float,
    dense_weight: float | None = None,
    top_k: int = 5,
    filters: dict | None = None,
):
    """
    Aaya adaptive retrieval wrapper.

    This does not re-implement Salma's retriever.
    It uses Salma's hybrid retriever as the baseline dependency, then applies
    Aaya's online/adaptive BM25 weighting around it.

    It supports two possible HybridRetriever versions:
    1. search(..., bm25_weight=...) is supported.
    2. search(...) does not accept bm25_weight, but the object has self.bm25_weight.
    """

    # For Aaya adaptation, weighted fusion is required.
    # If Salma's hybrid object supports a normalization attribute, force minmax
    # because RRF ignores BM25/dense weights.
    if hasattr(hybrid, "normalization"):
        hybrid.normalization = "minmax"

    search_params = inspect.signature(hybrid.search).parameters

    # Version 1: hybrid.search supports bm25_weight directly.
    if "bm25_weight" in search_params:
        return hybrid.search(
            query=query,
            k=top_k,
            filters=filters,
            bm25_weight=bm25_weight,
        )

    # Version 2: Salma's baseline retriever does not accept bm25_weight
    # as a parameter, but stores the weight in hybrid.bm25_weight.
    if hasattr(hybrid, "bm25_weight"):
        old_weight = hybrid.bm25_weight
        hybrid.bm25_weight = bm25_weight

        try:
            # Some versions accept filters; some may not.
            if "filters" in search_params:
                return hybrid.search(
                    query=query,
                    k=top_k,
                    filters=filters,
                )

            return hybrid.search(
                query=query,
                k=top_k,
            )

        finally:
            # Restore so each experiment row controls its own weight.
            hybrid.bm25_weight = old_weight

    raise TypeError(
        "The loaded HybridRetriever does not support adaptive weighting. "
        "It must either accept bm25_weight in search(), or expose hybrid.bm25_weight."
    )

In [10]:
def get_result_id(result):
    return (
        result.get("doc_id")
        or result.get("chunk_id")
        or result.get("id")
        or result.get("source")
        or result.get("metadata", {}).get("source")
    )

In [11]:
TOPIC_ALIASES = {
    "policy and governance": "policy_governance",
    "policy_governance": "policy_governance",

    "mitigation": "mitigation",
    "adaptation": "adaptation",

    "climate science": "climate_science",
    "climate_science": "climate_science",

    "technology and innovation": "technology_innovation",
    "technology innovation": "technology_innovation",
    "technology_innovation": "technology_innovation",

    "uae cop28": "uae_cop28",
    "uae_cop28": "uae_cop28",
    "cop28": "uae_cop28",

    "sustainability": "sustainability",
}


def normalize_topic(topic):
    if topic is None:
        return "unknown"

    topic = str(topic).strip().lower()
    topic = topic.replace("-", " ")
    topic = topic.replace("_", " ")

    return TOPIC_ALIASES.get(topic, topic.replace(" ", "_"))

In [12]:
# 9. Retrieval metrics

def result_matches_topic(result, relevant_topic: str) -> bool:
    relevant_topic = normalize_topic(relevant_topic)
    return relevant_topic in get_result_topics(result)


def recall_at_k(results, relevant_topic: str, k: int = 5) -> float:
    return float(any(result_matches_topic(r, relevant_topic) for r in results[:k]))


def mrr_at_k(results, relevant_topic: str, k: int = 5) -> float:
    for i, r in enumerate(results[:k], start=1):
        if result_matches_topic(r, relevant_topic):
            return 1.0 / i
    return 0.0


def ndcg_at_k(results, relevant_topic: str, k: int = 5) -> float:
    gains = [1.0 if result_matches_topic(r, relevant_topic) else 0.0 for r in results[:k]]
    dcg = sum(g / math.log2(i + 2) for i, g in enumerate(gains))

    ideal_gains = sorted(gains, reverse=True)
    idcg = sum(g / math.log2(i + 2) for i, g in enumerate(ideal_gains))

    return dcg / idcg if idcg > 0 else 0.0


def citation_hit_rate_proxy(results, relevant_topic: str) -> float:
    return float(len(results) > 0 and result_matches_topic(results[0], relevant_topic))

In [13]:
# 10. Retrieval router: static, topic-gated, adaptive
# Improved version:
# - uses confidence gating for adaptive routing
# - uses smaller safer feedback steps
# - updates only after repeated failures for the same predicted topic

class RetrievalRouter:
    def __init__(
        self,
        mode: str,
        min_confidence: float = 0.45,
        failures_before_update: int = 2,
    ):
        if mode not in {"static", "topic_gated", "adaptive"}:
            raise ValueError("mode must be one of: static, topic_gated, adaptive")

        self.mode = mode
        self.min_confidence = min_confidence
        self.failures_before_update = failures_before_update

        # Safer than step=0.05 and 0.10-0.90.
        # This prevents adaptive retrieval from becoming too BM25-heavy or too dense-heavy.
        self.feedback_adapter = FeedbackAdapter(
            default_bm25_weight=0.50,
            step=0.02,
            min_weight=0.25,
            max_weight=0.75,
        )

        self.failure_counts = {}

    def topic_confidence(self, classifier, query: str, predicted_topic: Optional[str]) -> float:
        if predicted_topic is None:
            return 0.0

        try:
            probs = classifier.predict_proba(query)
        except Exception:
            return 0.0

        if not probs:
            return 0.0

        return float(probs.get(predicted_topic, 0.0))

    def choose_weights(
        self,
        predicted_topic: Optional[str],
        classifier=None,
        query: Optional[str] = None,
    ):
        if self.mode == "static":
            return {
                **STATIC_WEIGHTS.copy(),
                "topic_confidence": None,
                "routing_reason": "static_baseline",
            }

        if self.mode == "topic_gated":
            return {
                **TOPIC_ROUTING_PROFILES.get(predicted_topic, STATIC_WEIGHTS).copy(),
                "topic_confidence": None,
                "routing_reason": "fixed_topic_profile",
            }

        # Adaptive mode
        confidence = self.topic_confidence(classifier, query, predicted_topic)

        # If River is uncertain, do not adapt. Fall back to static.
        if predicted_topic is None or confidence < self.min_confidence:
            return {
                **STATIC_WEIGHTS.copy(),
                "topic_confidence": confidence,
                "routing_reason": "fallback_static_low_confidence",
            }

        weights = self.feedback_adapter.get_weights(predicted_topic)

        return {
            **weights,
            "topic_confidence": confidence,
            "routing_reason": "adaptive_confident_topic",
        }

    def update_from_feedback(
        self,
        helpful: bool,
        predicted_topic: Optional[str],
        reason: str,
        routing_reason: Optional[str] = None,
    ):
        if self.mode != "adaptive":
            return {
                "feedback_update_applied": False,
                "updated_bm25_weight": None,
                "updated_dense_weight": None,
                "update_note": "not_adaptive_mode",
            }

        if routing_reason != "adaptive_confident_topic":
            return {
                "feedback_update_applied": False,
                "updated_bm25_weight": None,
                "updated_dense_weight": None,
                "update_note": "skipped_low_confidence_or_static_fallback",
            }

        topic = predicted_topic or "global"

        # If helpful, reset the failure count and keep weights stable.
        if helpful:
            self.failure_counts[topic] = 0
            return {
                "feedback_update_applied": False,
                "updated_bm25_weight": self.feedback_adapter.get_weights(topic)["bm25_weight"],
                "updated_dense_weight": self.feedback_adapter.get_weights(topic)["dense_weight"],
                "update_note": "helpful_no_weight_change",
            }

        # Only adapt after repeated failures for the same topic.
        self.failure_counts[topic] = self.failure_counts.get(topic, 0) + 1

        if self.failure_counts[topic] < self.failures_before_update:
            return {
                "feedback_update_applied": False,
                "updated_bm25_weight": self.feedback_adapter.get_weights(topic)["bm25_weight"],
                "updated_dense_weight": self.feedback_adapter.get_weights(topic)["dense_weight"],
                "update_note": "failure_buffered_no_update_yet",
            }

        update = self.feedback_adapter.update(
            helpful=False,
            query_topic=topic,
            reason=reason,
        )

        self.failure_counts[topic] = 0

        return {
            "feedback_update_applied": True,
            "updated_bm25_weight": update.bm25_weight,
            "updated_dense_weight": update.dense_weight,
            "update_note": f"updated_after_{self.failures_before_update}_failures",
        }

In [14]:
# 11. Generate an evaluation stream using the existing River D1 stream generator

stream = river_mod.generate_climate_query_stream(
    n_queries=600,
    drift_at=350,
    seed=42,
    drift_window=90,
)

pd.DataFrame([e.__dict__ for e in stream[:5]])

In [15]:
sample_results = retrieve_hybrid(
    query="What did COP28 say about fossil fuels?",
    bm25_weight=0.65,
    top_k=1,
)

print(sample_results[0])
print(sample_results[0].keys())

{'chunk_id': 'chunk_044222', 'doc_number': 'D262', 'document_id': 'pascual_2023_diverse_values_nature_sustainability_w4385691013', 'title': 'Diverse values of nature for sustainability', 'text': 'licy information about studies involving animals; ARRIVE guidelines recommended for reporting animal research, and Sex and Gender in Research Laboratory animals For laboratory animals, report species, strain and age OR state that the study did not involve laboratory animals. Wild animals Provide details on animals observed in or captured in the field; report species and age where possible. Describe how animals were caught and transported and what happened to captive animals after the study (if killed, explain why and describe method; if released, say where and when) OR state that the study did not involve wild animals. Reporting on sex Indicate if findings apply to only one sex; describe whe', 'page_start': 16, 'page_end': 16, 'topics': ['policy and governance', 'sustainability'], 'countries':

In [16]:
def get_result_topics(result):
    """
    Return all normalized topics from a real retrieval result.
    Your chunks use: topics: ['policy and governance', 'sustainability']
    """
    raw_topics = []

    if result.get("topic"):
        raw_topics.append(result["topic"])

    if result.get("topics"):
        if isinstance(result["topics"], list):
            raw_topics.extend(result["topics"])
        else:
            raw_topics.append(result["topics"])

    metadata = result.get("metadata", {})
    if isinstance(metadata, dict):
        if metadata.get("topic"):
            raw_topics.append(metadata["topic"])

        if metadata.get("topics"):
            if isinstance(metadata["topics"], list):
                raw_topics.extend(metadata["topics"])
            else:
                raw_topics.append(metadata["topics"])

    normalized = [normalize_topic(t) for t in raw_topics if t]
    return normalized or ["unknown"]


def get_result_topic(result):
    """
    Keep single-topic compatibility by returning the first normalized topic.
    """
    return get_result_topics(result)[0]


def get_result_id(result):
    """
    Safely extract document/chunk id from real retrieval result.
    """
    return (
        result.get("doc_id")
        or result.get("chunk_id")
        or result.get("id")
        or result.get("source")
        or result.get("metadata", {}).get("source")
        or "unknown"
    )

In [17]:
# 12. Run prequential retrieval evaluation
# Important order: predict topic -> route retrieval -> score retrieval -> update feedback -> learn topic.



def make_classifier():
    return river_mod.RiverTopicClassifier()

def feedback_reason(results, actual_topic: str) -> tuple[bool, str]:
    """
    Improved proxy feedback.

    Before:
    - marked helpful only if top-1 matched the topic.

    Now:
    - marks helpful if any result in top-5 matches the topic.
    - this matches the Recall@5 evaluation and avoids noisy punishment.
    """
    if not results:
        return False, "no_results"

    # If the correct topic appears anywhere in top-5, retrieval was partially useful.
    if recall_at_k(results, actual_topic, k=5) > 0:
        if result_matches_topic(results[0], actual_topic):
            return True, "helpful_top1"
        return True, "helpful_in_top5"

    actual_topic = normalize_topic(actual_topic)

    if actual_topic in {"policy_governance", "uae_cop28"}:
        return False, "missed_exact_policy"

    return False, "keyword_mismatch"

def evaluate_method(mode: str, stream, top_k: int = 5):
    classifier = make_classifier()
    router = RetrievalRouter(mode)
    rows = []
    n_topic_correct = 0

    for ex in stream:
        start = time.perf_counter()

        predicted_topic = classifier.predict(ex.query)

        weights = router.choose_weights(
            predicted_topic=predicted_topic,
            classifier=classifier,
            query=ex.query,
        )

        results = retrieve_hybrid(
            ex.query,
            bm25_weight=weights["bm25_weight"],
            dense_weight=weights["dense_weight"],
            top_k=top_k,
        )

        retrieval_recall = recall_at_k(results, ex.topic, k=top_k)
        retrieval_mrr = mrr_at_k(results, ex.topic, k=top_k)
        retrieval_ndcg = ndcg_at_k(results, ex.topic, k=top_k)
        citation_hit = citation_hit_rate_proxy(results, ex.topic)

        helpful, reason = feedback_reason(results, ex.topic)

        update_info = router.update_from_feedback(
            helpful=helpful,
            predicted_topic=predicted_topic,
            reason=reason,
            routing_reason=weights.get("routing_reason"),
        )

        classifier.learn(ex.query, ex.topic)
        topic_correct = predicted_topic == ex.topic
        n_topic_correct += int(topic_correct)

        latency_ms = (time.perf_counter() - start) * 1000

        rows.append({
            "method": mode,
            "index": ex.index,
            "phase": ex.phase,
            "query": ex.query,
            "actual_topic": ex.topic,
            "predicted_topic": predicted_topic or "__none__",
            "topic_correct": topic_correct,
            "bm25_weight": weights["bm25_weight"],
            "dense_weight": weights["dense_weight"],
            "top_doc_id": get_result_id(results[0]) if results else None,
            "top_doc_topic": ", ".join(get_result_topics(results[0])) if results else None,
            "recall@5": retrieval_recall,
            "mrr@5": retrieval_mrr,
            "ndcg@5": retrieval_ndcg,
            "citation_hit": citation_hit,
            "feedback_helpful": helpful,
            "feedback_reason": reason,
            "topic_confidence": weights.get("topic_confidence"),
            "routing_reason": weights.get("routing_reason"),
            "feedback_update_applied": update_info.get("feedback_update_applied"),
            "updated_bm25_weight": update_info.get("updated_bm25_weight"),
            "updated_dense_weight": update_info.get("updated_dense_weight"),
            "update_note": update_info.get("update_note"),
            "prequential_topic_accuracy_so_far": n_topic_correct / ex.index,
            "latency_ms": latency_ms,
        })

    return pd.DataFrame(rows)

all_results = pd.concat([
    evaluate_method("static", stream),
    evaluate_method("topic_gated", stream),
    evaluate_method("adaptive", stream),
], ignore_index=True)

all_results.head()

In [18]:
# 13. Create Aaya online-adaptation metrics summary table

summary = (
    all_results
    .groupby("method")
    .agg(
        **{
            "Recall@5": ("recall@5", "mean"),
            "MRR@5": ("mrr@5", "mean"),
            "nDCG@5": ("ndcg@5", "mean"),
            "Answer citation hit rate": ("citation_hit", "mean"),
            "Prequential topic accuracy": ("topic_correct", "mean"),
            "Mean latency ms": ("latency_ms", "mean"),
            "Final mean BM25 weight": ("bm25_weight", "mean"),
            "Final mean dense weight": ("dense_weight", "mean"),
        }
    )
    .reset_index()
)

summary.insert(1, "Topic source", summary["method"].map({
    "static": "none",
    "topic_gated": "River prediction",
    "adaptive": "River prediction",
}))
summary.insert(2, "BM25 weight source", summary["method"].map({
    "static": "fixed 0.50 / 0.50",
    "topic_gated": "fixed topic profile",
    "adaptive": "FeedbackAdapter per-topic weights",
}))
summary["Notes"] = summary["method"].map({
    "static": "Salma hybrid retriever baseline",
    "topic_gated": "Aaya River topic routing on hybrid baseline",
    "adaptive": "Aaya River + FeedbackAdapter online adaptation",
})

summary = summary.round(4)
summary

In [19]:
# 14. Phase-level table: useful for showing adaptation after drift

phase_summary = (
    all_results
    .groupby(["method", "phase"])
    .agg(
        **{
            "Recall@5": ("recall@5", "mean"),
            "MRR@5": ("mrr@5", "mean"),
            "nDCG@5": ("ndcg@5", "mean"),
            "Citation hit": ("citation_hit", "mean"),
            "Topic accuracy": ("topic_correct", "mean"),
            "Mean BM25": ("bm25_weight", "mean"),
        }
    )
    .reset_index()
    .round(4)
)
phase_summary

In [20]:
# 15. Save Aaya D2 online-adaptation outputs
#
# Important boundary:
# These are Aaya online/adaptive routing outputs, not Salma's retrieval-comparison outputs.
# Salma's retrieval comparison remains separate, e.g. d2_search_metrics.csv.

reports_tables_dir = PROJECT_ROOT / "reports" / "tables"
reports_tables_dir.mkdir(parents=True, exist_ok=True)

all_results_path = reports_tables_dir / "d2_online_adaptation_prequential_results.csv"
summary_path = reports_tables_dir / "d2_online_adaptation_summary.csv"
phase_summary_path = reports_tables_dir / "d2_online_adaptation_phase_summary.csv"

all_results.to_csv(all_results_path, index=False)
summary.to_csv(summary_path, index=False)
phase_summary.to_csv(phase_summary_path, index=False)

print("Saved Aaya online-adaptation outputs:")
print(all_results_path)
print(summary_path)
print(phase_summary_path)


Saved Aaya online-adaptation outputs:
D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent\reports\tables\d2_online_adaptation_prequential_results.csv
D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent\reports\tables\d2_online_adaptation_summary.csv
D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent\reports\tables\d2_online_adaptation_phase_summary.csv


In [21]:
# 16. Optional safe cleanup of old/ambiguous Aaya CSV filenames
#
# This cell is intentionally conservative.
# It does NOT delete anything unless DELETE_OLD_AAYA_CSVS is set to True.
# It only targets exact old Aaya filenames that were renamed to avoid confusion with Salma's retrieval outputs.

DELETE_OLD_AAYA_CSVS = False

old_aaya_csv_names = [
    "d2_prequential_retrieval_results.csv",
    "d2_retrieval_metrics_summary.csv",
    "d2_phase_metrics_summary.csv",
]

print("Old Aaya CSV cleanup mode:", "DELETE" if DELETE_OLD_AAYA_CSVS else "DRY RUN ONLY")
print("Only these exact old Aaya filenames are considered:")
for name in old_aaya_csv_names:
    print("-", name)

for name in old_aaya_csv_names:
    path = reports_tables_dir / name
    if path.exists():
        if DELETE_OLD_AAYA_CSVS:
            path.unlink()
            print("Deleted old Aaya CSV:", path)
        else:
            print("Would delete if DELETE_OLD_AAYA_CSVS=True:", path)
    else:
        print("Not found:", path)


Old Aaya CSV cleanup mode: DRY RUN ONLY
Only these exact old Aaya filenames are considered:
- d2_prequential_retrieval_results.csv
- d2_retrieval_metrics_summary.csv
- d2_phase_metrics_summary.csv
Would delete if DELETE_OLD_AAYA_CSVS=True: D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent\reports\tables\d2_prequential_retrieval_results.csv
Would delete if DELETE_OLD_AAYA_CSVS=True: D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent\reports\tables\d2_retrieval_metrics_summary.csv
Would delete if DELETE_OLD_AAYA_CSVS=True: D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent\reports\tables\d2_phase_metrics_summary.csv


In [22]:
# 17. Required Aaya output: static hybrid baseline vs online adaptive routing

online_vs_static = summary[summary["method"].isin(["static", "adaptive"])].copy()

online_vs_static_path = reports_tables_dir / "d2_online_vs_static.csv"
online_vs_static.to_csv(online_vs_static_path, index=False)

print("Saved required Aaya D2 comparison table:")
print(online_vs_static_path)
online_vs_static


Saved required Aaya D2 comparison table:
D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent\reports\tables\d2_online_vs_static.csv


In [23]:
# 17. Where adaptive helps and hurts compared with static

static_rows = (
    all_results[all_results["method"] == "static"]
    .set_index("index")
    .sort_index()
)

adaptive_rows = (
    all_results[all_results["method"] == "adaptive"]
    .set_index("index")
    .sort_index()
)

helps_hurts_rows = []

for metric_col in ["recall@5", "mrr@5", "ndcg@5", "citation_hit"]:
    diff = adaptive_rows[metric_col] - static_rows[metric_col]

    helps_hurts_rows.append({
        "metric": metric_col,
        "static_mean": static_rows[metric_col].mean(),
        "adaptive_mean": adaptive_rows[metric_col].mean(),
        "adaptive_minus_static": diff.mean(),
        "adaptive_helped_queries": int((diff > 0).sum()),
        "adaptive_hurt_queries": int((diff < 0).sum()),
        "same_queries": int((diff == 0).sum()),
    })

helps_hurts_table = pd.DataFrame(helps_hurts_rows).round(4)

helps_hurts_path = reports_tables_dir / "d2_adaptive_helps_hurts.csv"
helps_hurts_table.to_csv(helps_hurts_path, index=False)

print("Saved helps/hurts table:")
print(helps_hurts_path)

helps_hurts_table

Saved helps/hurts table:
D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent\reports\tables\d2_adaptive_helps_hurts.csv


In [24]:
# 18. Latency cost of adaptive retrieval

latency_cost = pd.DataFrame([{
    "static_mean_latency_ms": static_rows["latency_ms"].mean(),
    "adaptive_mean_latency_ms": adaptive_rows["latency_ms"].mean(),
    "adaptive_minus_static_ms": (
        adaptive_rows["latency_ms"].mean()
        - static_rows["latency_ms"].mean()
    ),
}]).round(4)

latency_cost_path = reports_tables_dir / "d2_latency_cost.csv"
latency_cost.to_csv(latency_cost_path, index=False)

print("Saved latency cost table:")
print(latency_cost_path)

latency_cost

Saved latency cost table:
D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent\reports\tables\d2_latency_cost.csv


In [25]:
# 19. Feedback-triggered adaptation evidence

adaptive_log = (
    all_results[all_results["method"] == "adaptive"]
    .copy()
    .sort_values("index")
)

feedback_update_evidence = adaptive_log[
    adaptive_log["feedback_update_applied"] == True
][[
    "index",
    "phase",
    "query",
    "actual_topic",
    "predicted_topic",
    "topic_confidence",
    "routing_reason",
    "bm25_weight",
    "dense_weight",
    "updated_bm25_weight",
    "updated_dense_weight",
    "feedback_helpful",
    "feedback_reason",
    "update_note",
    "top_doc_id",
    "top_doc_topic",
    "recall@5",
    "citation_hit",
    "latency_ms",
]]

feedback_update_path = reports_tables_dir / "d2_feedback_update_evidence.csv"
feedback_update_evidence.to_csv(feedback_update_path, index=False)

print("Saved feedback-triggered update evidence:")
print(feedback_update_path)
print("Number of adaptive feedback updates:", len(feedback_update_evidence))

feedback_update_evidence.head(10)

Saved feedback-triggered update evidence:
D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent\reports\tables\d2_feedback_update_evidence.csv
Number of adaptive feedback updates: 110


In [26]:
# 20. Example where online adaptation changes retrieval behavior

behavior_compare = adaptive_rows.join(
    static_rows,
    lsuffix="_adaptive",
    rsuffix="_static",
)

behavior_change_examples = behavior_compare[
    behavior_compare["top_doc_id_adaptive"]
    != behavior_compare["top_doc_id_static"]
].copy()

# Prefer examples where adaptive improves citation hit.
improved_behavior_examples = behavior_change_examples[
    behavior_change_examples["citation_hit_adaptive"]
    > behavior_change_examples["citation_hit_static"]
].copy()

if len(improved_behavior_examples) > 0:
    example_table = improved_behavior_examples
else:
    example_table = behavior_change_examples

example_columns = [
    "query_adaptive",
    "actual_topic_adaptive",
    "predicted_topic_adaptive",
    "bm25_weight_static",
    "bm25_weight_adaptive",
    "dense_weight_static",
    "dense_weight_adaptive",
    "top_doc_id_static",
    "top_doc_topic_static",
    "citation_hit_static",
    "recall@5_static",
    "top_doc_id_adaptive",
    "top_doc_topic_adaptive",
    "citation_hit_adaptive",
    "recall@5_adaptive",
    "feedback_reason_adaptive",
    "latency_ms_static",
    "latency_ms_adaptive",
]

online_change_example = example_table[example_columns].head(1)

online_change_example_path = reports_tables_dir / "d2_online_behavior_change_example.csv"
online_change_example.to_csv(online_change_example_path, index=False)

print("Saved online behavior-change example:")
print(online_change_example_path)

online_change_example

Saved online behavior-change example:
D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent\reports\tables\d2_online_behavior_change_example.csv


In [27]:
# 21. Auto-generate short interpretation from actual numbers

static_summary = summary[summary["method"] == "static"].iloc[0]
adaptive_summary = summary[summary["method"] == "adaptive"].iloc[0]

recall_diff = adaptive_summary["Recall@5"] - static_summary["Recall@5"]
mrr_diff = adaptive_summary["MRR@5"] - static_summary["MRR@5"]
ndcg_diff = adaptive_summary["nDCG@5"] - static_summary["nDCG@5"]
citation_diff = adaptive_summary["Answer citation hit rate"] - static_summary["Answer citation hit rate"]
latency_diff = adaptive_summary["Mean latency ms"] - static_summary["Mean latency ms"]

print("D2 online retrieval adaptation interpretation")
print("------------------------------------------------")
print(f"Adaptive - Static Recall@5: {recall_diff:.4f}")
print(f"Adaptive - Static MRR@5: {mrr_diff:.4f}")
print(f"Adaptive - Static nDCG@5: {ndcg_diff:.4f}")
print(f"Adaptive - Static citation hit rate: {citation_diff:.4f}")
print(f"Adaptive - Static latency cost: {latency_diff:.4f} ms")
print()
print(f"Feedback-triggered BM25 changes detected: {len(feedback_update_evidence)}")
print(f"Behavior-change examples detected: {len(behavior_change_examples)}")

D2 online retrieval adaptation interpretation
------------------------------------------------
Adaptive - Static Recall@5: 0.0033
Adaptive - Static MRR@5: -0.0020
Adaptive - Static nDCG@5: 0.0012
Adaptive - Static citation hit rate: 0.0016
Adaptive - Static latency cost: -24.3568 ms

Feedback-triggered BM25 changes detected: 110
Behavior-change examples detected: 127


**Evaluation limitation - noisy topic labels**

The adaptive feedback signal is estimated using topic-level relevance. This is useful for D2 because no real user feedback is available, but it is weaker than evidence-level relevance. A retrieved chunk may be useful even if its topic label differs from the expected topic, and a chunk with the same topic may still not answer the query. Noisy or broad topic labels can therefore cause the `FeedbackAdapter` to update BM25/dense weights in the wrong direction. This may underestimate the true value of online adaptation. A stronger D3 evaluation should use chunk-level or document-level gold relevance, such as whether the retrieved `chunk_id` or `document_id` matches the expected evidence.
